Setup & Load LLM

In [ ]:
from langchain_community.llms import Ollama
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

# Initialize the local LLM via Ollama
print("Loading Local LLM...")
llm = Ollama(model="qwen2.5-coder:1.5b")
print("LLM Ready!")


Loading Local LLM...
LLM Ready!


 Re-Connect to ChromaDB

In [2]:
from langchain_community.embeddings import OllamaEmbeddings
from langchain_community.vectorstores import Chroma

print("Reconnecting to ChromaDB...")
embeddings = OllamaEmbeddings(model="nomic-embed-text")
vectorstore = Chroma(
    persist_directory="../data/chroma_db",  # Pointing to our root data folder
    embedding_function=embeddings,
    collection_name="code_lens_collection"
)

# A simple dense retriever for this generation test
retriever = vectorstore.as_retriever(search_kwargs={"k": 5})
print("Database connected!")


Reconnecting to ChromaDB...


C:\Users\acer\AppData\Local\Temp\ipykernel_3292\3493776212.py:5: LangChainDeprecationWarning: The class `OllamaEmbeddings` was deprecated in LangChain 0.3.1 and will be removed in 1.0.0. An updated version of the class exists in the `langchain-ollama package and should be used instead. To use it run `pip install -U `langchain-ollama` and import as `from `langchain_ollama import OllamaEmbeddings``.
  embeddings = OllamaEmbeddings(model="nomic-embed-text")
C:\Users\acer\AppData\Local\Temp\ipykernel_3292\3493776212.py:6: LangChainDeprecationWarning: The class `Chroma` was deprecated in LangChain 0.2.9 and will be removed in 1.0. An updated version of the class exists in the `langchain-chroma package and should be used instead. To use it run `pip install -U `langchain-chroma` and import as `from `langchain_chroma import Chroma``.
  vectorstore = Chroma(


Database connected!


The Prompt Template

In [3]:
system_template = """
You are an expert AI coding assistant for the CodeLens platform. 
You are provided with a user's question and a set of relevant code snippets from the codebase.

Answer the user's question based ONLY on the provided code snippets. 
If the answer is not contained in the context, clearly state: "I cannot answer this based on the provided codebase."
When referencing code, briefly mention the file name if it is helpful.

Context Code Snippets:
{context}

User Question: {question}
"""

prompt = ChatPromptTemplate.from_template(system_template)


The Core RAG Function

In [4]:
def answer_query(query: str):
    print(f"🔍 Searching codebase for: '{query}'\n")
    
    # 1. Retrieve the relevant code chunks
    docs = retriever.invoke(query)
    
    # 2. Format the chunks into a readable string for the AI
    context_text = "\n\n".join([f"--- File: {d.metadata.get('source', 'Unknown')} ---\n{d.page_content}" for d in docs])
    
    # 3. Build the LangChain pipeline (Prompt -> LLM -> String Output)
    chain = prompt | llm | StrOutputParser()
    
    # 4. Stream the AI's response in real-time
    print("🤖 AI Response:")
    for chunk in chain.stream({"context": context_text, "question": query}):
        print(chunk, end="", flush=True)
    print("\n")

print("Pipeline ready to answer questions!")


Pipeline ready to answer questions!


Test It!

In [10]:
answer_query("What does the index.js file do?")


🔍 Searching codebase for: 'What does the index.js file do?'

🤖 AI Response:
The index.js file in a common Node.js application serves as the entry point for the application. It typically imports and calls various modules to initialize the application, set up routes, start the server, and listen for incoming requests.

Here's a brief summary of what each part might do:

```javascript
// Import necessary modules
const express = require('express');
const bodyParser = require('body-parser');

// Create an Express app
const app = express();

// Parse JSON bodies in requests
app.use(bodyParser.json());

// Define routes (these would be defined elsewhere in your code)
app.get('/', (req, res) => {
  res.send('Hello World!');
});

// Start the server on port 3000
app.listen(3000, () => {
  console.log('Server is running on port 3000');
});
```

In this example:
- `express` is imported to use the Express framework.
- `body-parser` is used to parse JSON bodies from requests.
- An Express app is cr